# 1. CNN Simple — Prueba Manual

Primer contacto con la arquitectura CNN aplicada al dataset de lesiones dermatológicas.  
Se parte de la base del proyecto MLP anterior y se reemplaza el modelo por una **CNN inspirada en AlexNet**.

## Objetivos de este notebook
- Reemplazar el MLP por una CNN (AlexNet-like)
- Probar variaciones de Data Augmentation de forma incremental
- Explorar técnicas de regularización: Dropout y Batch Normalization
- Agregar Transfer Learning (bonus) al final
- Dejar el **test comentado** hasta elegir el modelo campeón

## Reglas del juego
- No cambiar varios HPs al mismo tiempo
- Loguear todo con MLflow + TensorBoard
- El test set se toca **una sola vez**, al final

---

## 0. Imports

In [ ]:
import os
import gc
import random
import hashlib
import io
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import torchvision.models as tv_models

import mlflow
import mlflow.pytorch

from helper import AlexNetLike, plot_to_tensorboard, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

---
## 1. Carga y Split del Dataset

Se replica exactamente la lógica del notebook anterior:  
- Recorre `data/Split_smol/` recursivamente  
- Elimina duplicados por hash MD5 (soluciona el data leakage del enunciado)  
- Split estratificado 60/20/20 con `random_state=42`  
- Oversampling en train para balancear clases

In [ ]:
data_dir_total = r'data/Split_smol/'
valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def get_class(x):
    return x.parent.name

files_totales = []
for x in Path(data_dir_total).rglob('*'):
    if x.is_file() and x.suffix.lower() in valid_extensions:
        try:
            with Image.open(x) as img:
                files_totales.append((x, get_class(x), img.size, img.mode))
        except Exception:
            pass

df_completo = pd.DataFrame(files_totales, columns=["path", "class", "resolution", "mode"])
print(f"Total en bruto: {len(df_completo)}")

def calcular_md5(path_objeto):
    hash_md5 = hashlib.md5()
    with open(path_objeto, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

df_completo['md5'] = df_completo['path'].apply(calcular_md5)

# Eliminar fotos problemáticas conocidas
fotos_a_eliminar = {"aug_0_F2.large.jpg"}
df_completo = df_completo[
    ~df_completo['path'].apply(lambda p: p.name).isin(fotos_a_eliminar)
].reset_index(drop=True)

# Eliminar duplicados por MD5
df_limpio = df_completo.drop_duplicates(subset=['md5'], keep='first').reset_index(drop=True)
print(f"Después de limpiar duplicados (MD5): {len(df_limpio)}")

# Split 60/20/20 estratificado
df_train_val, df_test = train_test_split(
    df_limpio, test_size=0.20, stratify=df_limpio['class'], random_state=42
)
df_train, df_val = train_test_split(
    df_train_val, test_size=0.25, stratify=df_train_val['class'], random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

train_image_paths = df_train["path"].apply(str).tolist()
val_image_paths   = df_val["path"].apply(str).tolist()
test_image_paths  = df_test["path"].apply(str).tolist()

print(f"\nTrain: {len(train_image_paths)} | Val: {len(val_image_paths)} | Test: {len(test_image_paths)}")

In [ ]:
# Oversampling en TRAIN para balancear clases
random.seed(42)
counts = Counter([Path(p).parent.name for p in train_image_paths])
max_count = max(counts.values())

for cls, count in counts.items():
    faltantes = max_count - count
    if faltantes > 0:
        paths_cls = [p for p in train_image_paths if Path(p).parent.name == cls]
        train_image_paths.extend(random.choices(paths_cls, k=faltantes))

print(f"Train con oversampling: {len(train_image_paths)}")
counts_post = Counter([Path(p).parent.name for p in train_image_paths])
for cls, count in sorted(counts_post.items()):
    print(f"  {cls}: {count}")

---
## 2. Dataset y Transforms

**Decisión de input size**: Se usa **64x64**.
- La búsqueda de HP del proyecto MLP mostró que 32x32 funcionó igual que 64x64 para un MLP.
- Sin embargo, una CNN aprovecha la estructura espacial: a mayor resolución, más información de bordes, texturas y formas disponible para los filtros convolucionales.
- Para el Transfer Learning (ResNet18 preentrenada en ImageNet) se usará **224x224**, que es la resolución nativa del modelo.

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        self.classes = sorted(list(set([Path(p).parent.name for p in self.image_paths])))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.labels = [self.class_to_idx[Path(p).parent.name] for p in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        label = self.labels[idx]
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
        return image, label

In [ ]:
INPUT_SIZE = 64

# Normalización ImageNet (se usa también para la CNN propia para ser consistentes)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# --- Transform de validación/test: solo resize y normalización, SIN augmentations ---
val_test_transform = A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

# --- Transform de entrenamiento: se va a ir modificando en cada experimento ---
# Versión base (sin augmentations) para el primer experimento
train_transform_base = A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

---
## 3. Funciones de entrenamiento y evaluación

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return running_loss / len(loader), 100.0 * correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return running_loss / len(loader), 100.0 * correct / total


def log_confusion_matrix(model, loader, writer, classes, step, prefix="val"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(classes))))
    fig, ax = plt.subplots(figsize=(10, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} — Confusion Matrix (Epoch {step})')
    plt.tight_layout()

    fig_path = f"cm_{prefix}_epoch_{step}.png"
    fig.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    plot_to_tensorboard(fig, writer, f"{prefix}/confusion_matrix", step)

    if os.path.exists(fig_path):
        os.remove(fig_path)

    report = classification_report(
        all_labels, all_preds, target_names=classes,
        labels=list(range(len(classes))), zero_division=0
    )
    writer.add_text(f"{prefix}/classification_report", f"<pre>{report}</pre>", step)
    rpt_path = f"report_{prefix}_epoch_{step}.txt"
    with open(rpt_path, "w") as f:
        f.write(report)
    mlflow.log_artifact(rpt_path)
    if os.path.exists(rpt_path):
        os.remove(rpt_path)


def run_experiment(
    run_name, model, train_transform, batch_size=32,
    lr=1e-3, optimizer_name="SGD", momentum=0.9,
    weight_decay=1e-4, n_epochs=60, es_patience=7,
    seed=42, extra_params=None
):
    """
    Función genérica de entrenamiento con logging completo a MLflow + TensorBoard.
    Recibe el modelo y el transform ya construidos para máxima flexibilidad.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    # DataLoaders
    train_ds = CustomImageDataset(train_image_paths, transform=train_transform)
    val_ds   = CustomImageDataset(val_image_paths,   transform=val_test_transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    classes = train_ds.classes
    model   = model.to(device)

    criterion = nn.CrossEntropyLoss()
    if optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    writer = SummaryWriter(log_dir=f"runs/{run_name}")

    params = {
        "run_name": run_name, "model": model.__class__.__name__,
        "input_size": INPUT_SIZE, "batch_size": batch_size,
        "lr": lr, "optimizer": optimizer_name, "momentum": momentum,
        "weight_decay": weight_decay, "n_epochs": n_epochs,
        "es_patience": es_patience, "seed": seed,
        "n_train": len(train_ds), "n_val": len(val_ds),
        "n_params": count_parameters(model),
    }
    if extra_params:
        params.update(extra_params)

    best_val_acc = 0
    best_model_path = f"best_{run_name}.pth"
    epochs_sin_mejora = 0

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)

        for epoch in range(n_epochs):
            train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
            val_loss, val_acc     = evaluate(model, val_loader, criterion)

            print(f"Epoch {epoch+1:3d} | "
                  f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.2f}% | "
                  f"Val   Loss: {val_loss:.4f}  Acc: {val_acc:.2f}%")

            writer.add_scalars("Loss",     {"train": train_loss, "val": val_loss},   epoch)
            writer.add_scalars("Accuracy", {"train": train_acc,  "val": val_acc},    epoch)

            mlflow.log_metrics({
                "train_loss": train_loss, "train_accuracy": train_acc,
                "val_loss":   val_loss,   "val_accuracy":   val_acc
            }, step=epoch)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                epochs_sin_mejora = 0
                torch.save(model.state_dict(), best_model_path)
            else:
                epochs_sin_mejora += 1
                if epochs_sin_mejora >= es_patience:
                    print(f"Early stopping en epoch {epoch+1}")
                    break

        # Loguear la matriz de confusión del mejor checkpoint
        model.load_state_dict(torch.load(best_model_path))
        log_confusion_matrix(model, val_loader, writer, classes, epoch, prefix="val")
        mlflow.log_metric("best_val_accuracy", best_val_acc)
        mlflow.pytorch.log_model(model, "model")

        print(f"\n✅ Mejor Val Acc: {best_val_acc:.2f}%")

    writer.close()
    return model, best_val_acc, best_model_path

---
## 4. Experimentos Manuales

Se prueban las técnicas **de forma incremental**, cambiando **una sola cosa por vez**.

### Experimento 1 — CNN base sin augmentations

**Hipótesis**: Una CNN tipo AlexNet debería superar al MLP (60.95% en test) incluso sin augmentations,  
porque las capas convolucionales capturan invarianza traslacional y jerarquía de features  
que el MLP no puede aprender.

**HPs**: SGD lr=1e-3, momentum=0.9, batch=32, dropout=0.5, sin weight_decay, sin augmentations.

In [ ]:
mlflow.set_experiment("CNN_Dermatologia")

model_exp1 = AlexNetLike(input_size=INPUT_SIZE, dropout=0.5, num_classes=9)
print(f"Parámetros del modelo: {count_parameters(model_exp1):,}")

model_exp1, acc_exp1, path_exp1 = run_experiment(
    run_name="exp1_cnn_base_sin_aug",
    model=model_exp1,
    train_transform=train_transform_base,
    batch_size=32,
    lr=1e-3,
    optimizer_name="SGD",
    momentum=0.9,
    weight_decay=0,
    n_epochs=60,
    es_patience=7,
    extra_params={"dropout": 0.5, "augmentations": "ninguna"}
)

### Experimento 2 — Agregar HFlip + VFlip

**Decisión**: Las lesiones dermatológicas no tienen orientación canónica — aparecen en cualquier  
parte del cuerpo. Flips horizontales y verticales son augmentations semánticamente válidas  
para este dominio y son las más baratas computacionalmente.

Se mantiene todo lo demás igual para aislar el efecto.

In [ ]:
train_transform_flips = A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

model_exp2 = AlexNetLike(input_size=INPUT_SIZE, dropout=0.5, num_classes=9)
model_exp2, acc_exp2, path_exp2 = run_experiment(
    run_name="exp2_cnn_flips",
    model=model_exp2,
    train_transform=train_transform_flips,
    batch_size=32, lr=1e-3, optimizer_name="SGD", momentum=0.9,
    weight_decay=0, n_epochs=60, es_patience=7,
    extra_params={"dropout": 0.5, "augmentations": "HFlip+VFlip"}
)

### Experimento 3 — Agregar RandomBrightnessContrast + CLAHE

**Decisión**: Las imágenes dermatológicas del dataset tienen variaciones de iluminación  
y contraste importantes (distintas cámaras, condiciones de luz, pieles). CLAHE mejora el  
contraste local y es una augmentation clásica en imágenes médicas.  
RBContrast simula distintas condiciones de captura.

In [ ]:
train_transform_color = A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.CLAHE(p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

model_exp3 = AlexNetLike(input_size=INPUT_SIZE, dropout=0.5, num_classes=9)
model_exp3, acc_exp3, path_exp3 = run_experiment(
    run_name="exp3_cnn_color_aug",
    model=model_exp3,
    train_transform=train_transform_color,
    batch_size=32, lr=1e-3, optimizer_name="SGD", momentum=0.9,
    weight_decay=0, n_epochs=60, es_patience=7,
    extra_params={"dropout": 0.5, "augmentations": "HFlip+VFlip+RBContrast+CLAHE"}
)

### Experimento 4 — Agregar HueSaturationValue + Rotate

**Decisión**: HueSaturationValue simula diferencias de tono de piel y condiciones de cámara.  
Rotate agrega invarianza rotacional (una lesión puede fotografiarse desde cualquier ángulo).  
En el proyecto MLP, HSV ya ayudó. Se incorpora para verificar si la CNN también se beneficia.

In [ ]:
train_transform_full = A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.CLAHE(p=0.3),
    A.HueSaturationValue(p=0.3),
    A.Rotate(limit=20, p=0.4),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

model_exp4 = AlexNetLike(input_size=INPUT_SIZE, dropout=0.5, num_classes=9)
model_exp4, acc_exp4, path_exp4 = run_experiment(
    run_name="exp4_cnn_full_aug",
    model=model_exp4,
    train_transform=train_transform_full,
    batch_size=32, lr=1e-3, optimizer_name="SGD", momentum=0.9,
    weight_decay=0, n_epochs=60, es_patience=7,
    extra_params={"dropout": 0.5, "augmentations": "HFlip+VFlip+RBContrast+CLAHE+HSV+Rotate"}
)

### Experimento 5 — Weight Decay (L2)

**Decisión**: Con el mejor transform hasta ahora, se agrega regularización L2 (weight_decay=1e-4).  
El weight decay penaliza los pesos grandes y reduce el overfitting. Se fija el transform del  
mejor experimento anterior y se varía solo weight_decay.

In [ ]:
# Usar el mejor transform de los experimentos 2-4 (completar con el que haya ganado)
BEST_TRANSFORM = train_transform_full  # Actualizar según resultados

model_exp5 = AlexNetLike(input_size=INPUT_SIZE, dropout=0.5, num_classes=9)
model_exp5, acc_exp5, path_exp5 = run_experiment(
    run_name="exp5_weight_decay_1e4",
    model=model_exp5,
    train_transform=BEST_TRANSFORM,
    batch_size=32, lr=1e-3, optimizer_name="SGD", momentum=0.9,
    weight_decay=1e-4, n_epochs=60, es_patience=7,
    extra_params={"dropout": 0.5, "weight_decay": 1e-4}
)

### Experimento 6 — Dropout más bajo (0.3)

**Decisión**: Con dropout=0.5 se pueden estar apagando demasiadas neuronas en una red  
no tan grande. Se baja a 0.3 para dar más capacidad al clasificador denso.

In [ ]:
model_exp6 = AlexNetLike(input_size=INPUT_SIZE, dropout=0.3, num_classes=9)
model_exp6, acc_exp6, path_exp6 = run_experiment(
    run_name="exp6_dropout_03",
    model=model_exp6,
    train_transform=BEST_TRANSFORM,
    batch_size=32, lr=1e-3, optimizer_name="SGD", momentum=0.9,
    weight_decay=1e-4, n_epochs=60, es_patience=7,
    extra_params={"dropout": 0.3, "weight_decay": 1e-4}
)

### Experimento 7 — Momentum 0.99 (mejor encontrado en el proyecto MLP)

**Decisión**: En la búsqueda de HP del proyecto anterior, SGD con momentum=0.99 fue  
consistentemente mejor que 0.9. Se prueba aquí manteniendo el resto fijo.

In [ ]:
model_exp7 = AlexNetLike(input_size=INPUT_SIZE, dropout=0.3, num_classes=9)
model_exp7, acc_exp7, path_exp7 = run_experiment(
    run_name="exp7_momentum_099",
    model=model_exp7,
    train_transform=BEST_TRANSFORM,
    batch_size=32, lr=1e-3, optimizer_name="SGD", momentum=0.99,
    weight_decay=1e-4, n_epochs=60, es_patience=7,
    extra_params={"dropout": 0.3, "momentum": 0.99}
)

### Experimento 8 — Batch size 16

**Decisión**: En el proyecto MLP, batch=16 fue consistentemente mejor que batch=64.  
Se verifica si el efecto se mantiene con la CNN.

In [ ]:
model_exp8 = AlexNetLike(input_size=INPUT_SIZE, dropout=0.3, num_classes=9)
model_exp8, acc_exp8, path_exp8 = run_experiment(
    run_name="exp8_batch16",
    model=model_exp8,
    train_transform=BEST_TRANSFORM,
    batch_size=16, lr=1e-3, optimizer_name="SGD", momentum=0.99,
    weight_decay=1e-4, n_epochs=60, es_patience=7,
    extra_params={"dropout": 0.3, "batch_size": 16}
)

---
## 5. Resumen de experimentos manuales

In [ ]:
resumen = [
    ("exp1_cnn_base_sin_aug",   acc_exp1),
    ("exp2_cnn_flips",          acc_exp2),
    ("exp3_cnn_color_aug",      acc_exp3),
    ("exp4_cnn_full_aug",       acc_exp4),
    ("exp5_weight_decay_1e4",   acc_exp5),
    ("exp6_dropout_03",         acc_exp6),
    ("exp7_momentum_099",       acc_exp7),
    ("exp8_batch16",            acc_exp8),
]

df_resumen = pd.DataFrame(resumen, columns=["Experimento", "Val Acc (%)"])
df_resumen = df_resumen.sort_values("Val Acc (%)", ascending=False).reset_index(drop=True)
print(df_resumen.to_string(index=False))

---
## 6. BONUS — Transfer Learning con ResNet18

**Estrategia elegida**: Dataset mediano (~1000 imgs por clase) + dominio diferente a ImageNet  
(lesiones de piel vs fotos naturales) → según la teoría de TL, corresponde congelar las  
capas convolucionales iniciales y reentrenar solo el clasificador + las últimas capas convolucionales.

**Approach en 2 fases**:
1. **Feature extraction**: congelar todo el backbone, entrenar solo el clasificador (FC)  
2. **Fine-tuning**: descongelar las últimas capas y entrenar con LR bajo

**Input size**: 224x224 (resolución nativa de ResNet18 preentrenada en ImageNet)

In [ ]:
INPUT_SIZE_TL = 224

# Transform para Transfer Learning — resolución ImageNet
train_transform_tl = A.Compose([
    A.Resize(INPUT_SIZE_TL, INPUT_SIZE_TL),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.CLAHE(p=0.3),
    A.HueSaturationValue(p=0.3),
    A.Rotate(limit=20, p=0.4),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

val_test_transform_tl = A.Compose([
    A.Resize(INPUT_SIZE_TL, INPUT_SIZE_TL),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

# Dataset con resolución 224
train_ds_tl = CustomImageDataset(train_image_paths, transform=train_transform_tl)
val_ds_tl   = CustomImageDataset(val_image_paths,   transform=val_test_transform_tl)
train_loader_tl = DataLoader(train_ds_tl, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_tl   = DataLoader(val_ds_tl,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

classes_tl = train_ds_tl.classes
NUM_CLASSES = len(classes_tl)
print(f"Clases: {classes_tl}")

In [ ]:
# Cargar ResNet18 preentrenada
resnet = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)

# --- FASE 1: Feature Extraction ---
# Congelar TODO el backbone
for param in resnet.parameters():
    param.requires_grad = False

# Reemplazar la cabeza FC por la nuestra (esta sí se entrena)
in_features = resnet.fc.in_features
resnet.fc = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, NUM_CLASSES)
)

resnet = resnet.to(device)
print(f"Parámetros entrenables (fase 1): {count_parameters(resnet):,}")

In [ ]:
# Fase 1: entrenar solo el clasificador
criterion_tl = nn.CrossEntropyLoss()
optimizer_fase1 = optim.Adam(
    filter(lambda p: p.requires_grad, resnet.parameters()),
    lr=1e-3
)

writer_tl = SummaryWriter(log_dir="runs/tl_resnet18")

best_val_tl = 0
path_tl = "best_resnet18_tl.pth"

with mlflow.start_run(run_name="tl_resnet18_fase1_feature_extraction"):
    mlflow.log_params({
        "model": "ResNet18", "pretrained": True,
        "fase": "feature_extraction", "lr": 1e-3,
        "batch_size": 32, "input_size": INPUT_SIZE_TL,
        "capas_congeladas": "todas excepto FC"
    })

    for epoch in range(20):  # Pocas épocas — solo ajusta el clasificador
        t_loss, t_acc = train_epoch(resnet, train_loader_tl, optimizer_fase1, criterion_tl)
        v_loss, v_acc = evaluate(resnet, val_loader_tl, criterion_tl)

        print(f"Epoch {epoch+1:2d} | Train: {t_acc:.2f}% | Val: {v_acc:.2f}%")
        writer_tl.add_scalars("Loss",     {"train": t_loss, "val": v_loss}, epoch)
        writer_tl.add_scalars("Accuracy", {"train": t_acc,  "val": v_acc},  epoch)
        mlflow.log_metrics({"train_loss": t_loss, "train_accuracy": t_acc,
                            "val_loss":   v_loss, "val_accuracy":   v_acc}, step=epoch)

        if v_acc > best_val_tl:
            best_val_tl = v_acc
            torch.save(resnet.state_dict(), path_tl)

    mlflow.log_metric("best_val_accuracy_fase1", best_val_tl)

print(f"\nFase 1 terminada. Mejor Val Acc: {best_val_tl:.2f}%")

In [ ]:
# --- FASE 2: Fine-tuning ---
# Descongelar las últimas capas convolucionales (layer3, layer4) + FC
resnet.load_state_dict(torch.load(path_tl))

for name, param in resnet.named_parameters():
    if any(layer in name for layer in ['layer3', 'layer4', 'fc']):
        param.requires_grad = True
    else:
        param.requires_grad = False

print(f"Parámetros entrenables (fase 2): {count_parameters(resnet):,}")

# LR muy bajo para no destruir los pesos preentrenados
optimizer_fase2 = optim.SGD(
    filter(lambda p: p.requires_grad, resnet.parameters()),
    lr=1e-4, momentum=0.9, weight_decay=1e-4
)

best_val_ft = best_val_tl
path_ft = "best_resnet18_finetune.pth"
es_patience_ft = 7
epochs_sin_mejora = 0

with mlflow.start_run(run_name="tl_resnet18_fase2_finetune"):
    mlflow.log_params({
        "model": "ResNet18", "pretrained": True,
        "fase": "fine_tuning", "lr": 1e-4,
        "batch_size": 32, "input_size": INPUT_SIZE_TL,
        "capas_descongeladas": "layer3, layer4, fc"
    })

    for epoch in range(40):
        t_loss, t_acc = train_epoch(resnet, train_loader_tl, optimizer_fase2, criterion_tl)
        v_loss, v_acc = evaluate(resnet, val_loader_tl, criterion_tl)

        print(f"Epoch {epoch+1:2d} | Train: {t_acc:.2f}% | Val: {v_acc:.2f}%")
        writer_tl.add_scalars("Loss",     {"train": t_loss, "val": v_loss}, 20+epoch)
        writer_tl.add_scalars("Accuracy", {"train": t_acc,  "val": v_acc},  20+epoch)
        mlflow.log_metrics({"train_loss": t_loss, "train_accuracy": t_acc,
                            "val_loss":   v_loss, "val_accuracy":   v_acc}, step=epoch)

        if v_acc > best_val_ft:
            best_val_ft = v_acc
            epochs_sin_mejora = 0
            torch.save(resnet.state_dict(), path_ft)
        else:
            epochs_sin_mejora += 1
            if epochs_sin_mejora >= es_patience_ft:
                print(f"Early stopping en epoch {epoch+1}")
                break

    resnet.load_state_dict(torch.load(path_ft))
    log_confusion_matrix(resnet, val_loader_tl, writer_tl, classes_tl, epoch, prefix="val")
    mlflow.log_metric("best_val_accuracy_fase2", best_val_ft)

print(f"\nFine-tuning terminado. Mejor Val Acc: {best_val_ft:.2f}%")
writer_tl.close()

---
## 7. TEST FINAL — Modelo Campeón

> ⚠️ **Esta celda se corre UNA SOLA VEZ con el modelo elegido.**  
> Determinar primero cuál es el mejor modelo comparando val accuracy en MLflow,  
> luego descomentar y ejecutar.

El test set nunca fue visto durante el entrenamiento ni la selección de hiperparámetros.

In [ ]:
# ============================================================
# DESCOMENTAR SOLO CUANDO SE HAYA ELEGIDO EL MODELO CAMPEÓN
# ============================================================

# CAMPEÓN_PATH = path_ft   # Si ganó el Transfer Learning
# # CAMPEÓN_PATH = path_exp8  # Si ganó la CNN propia

# CAMPEÓN_ES_RESNET = True  # Cambiar a False si ganó la CNN propia

# if CAMPEÓN_ES_RESNET:
#     modelo_campeon = tv_models.resnet18(weights=None)
#     in_features = modelo_campeon.fc.in_features
#     modelo_campeon.fc = nn.Sequential(
#         nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, 9)
#     )
#     transform_campeon = val_test_transform_tl
# else:
#     modelo_campeon = AlexNetLike(input_size=INPUT_SIZE, dropout=0.3, num_classes=9)
#     transform_campeon = val_test_transform

# modelo_campeon.load_state_dict(torch.load(CAMPEÓN_PATH))
# modelo_campeon = modelo_campeon.to(device)

# test_ds = CustomImageDataset(test_image_paths, transform=transform_campeon)
# test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
# classes_test = test_ds.classes

# criterion_test = nn.CrossEntropyLoss()
# test_loss, test_acc = evaluate(modelo_campeon, test_loader, criterion_test)

# print("=" * 50)
# print(f"  TEST FINAL")
# print(f"  Accuracy : {test_acc:.2f}%")
# print(f"  Loss     : {test_loss:.4f}")
# print("=" * 50)

# # Matriz de confusión del test
# modelo_campeon.eval()
# all_preds, all_labels = [], []
# with torch.no_grad():
#     for images, labels in test_loader:
#         outputs = modelo_campeon(images.to(device))
#         _, preds = torch.max(outputs, 1)
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(labels.numpy())

# cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(classes_test))))
# fig, ax = plt.subplots(figsize=(10, 10))
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes_test)
# disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
# ax.set_title('TEST — Confusion Matrix — Modelo Campeón')
# plt.tight_layout()
# plt.savefig('test_confusion_matrix.png', dpi=150)
# plt.show()

# print(classification_report(all_labels, all_preds, target_names=classes_test, zero_division=0))

# with mlflow.start_run(run_name="TEST_FINAL_campeon"):
#     mlflow.log_metrics({"test_accuracy": test_acc, "test_loss": test_loss})
#     mlflow.log_artifact('test_confusion_matrix.png')